# Import Libraries

In [1]:
%pip install torch torchvision tensorboard

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
from torchvision import transforms as T
from utils.set_seed import set_random_seed
from torch.utils.tensorboard import SummaryWriter
import pandas as pd

device = 'cpu'
if torch.backends.mps.is_available(): device = torch.device('mps')
elif torch.cuda.is_available(): device = torch.device('cuda')

In [4]:
print(device)

cuda


In [5]:
from weathernet import WeatherNet
from weathernetplusplus import WeatherNetPlusPlus
from mtl_weathernet import MtlWeatherNet
from weathernet_transformer import WeatherNetTransformer

from data.data import get_dataloaders
import utils.trainer as trainer

# Set Hyperparameters, Load Dataset

In [6]:
# Hyperparameters
# TODO: experiment with different hyperparameters
batch_size = 192
learning_rate = 0.001
epochs = 10

set_random_seed(42) # seed for reproducibility

In [7]:
# Load BDD100KPlus dataset
trainloader, valloader, testloader = get_dataloaders(dataset_name="Bdd100kPlus", batch_size=batch_size)

In [8]:
# Print dataset statistics
print(f"Number of training samples: {len(trainloader.dataset)}")
print(f"Number of validation samples: {len(valloader.dataset)}")
print(f"Number of test samples: {len(testloader.dataset)}")

# print batches
print(f"Number of batches in training set: {len(trainloader)}")
print(f"Number of batches in validation set: {len(valloader)}")
print(f"Number of batches in test set: {len(testloader)}")

Number of training samples: 70000
Number of validation samples: 10000
Number of test samples: 20000
Number of batches in training set: 365
Number of batches in validation set: 53
Number of batches in test set: 105


# Initialize WeatherNet Model

In [9]:
# model: WeatherNet = WeatherNet()
# model: WeatherNetPlusPlus = WeatherNetPlusPlus()
# model: MtlWeatherNet = MtlWeatherNet()
model: WeatherNetTransformer = WeatherNetTransformer()

# Print the model architecture
print(model)

# Define optimizer for the model
# Could explore SGD, Adam, AdamW, etc
optimizer = torch.optim.Adam(model.parameters())

WeatherNetTransformer(
  (backbone): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
        

In [10]:
saved_state = None # path to saved model state
if saved_state is not None:
    print(f"Loading saved state from {saved_state}")
    model.load_checkpoint(saved_state)

In [11]:
import os
# trainloader.num_workers = int(os.cpu_count())
# valloader.num_workers = int(os.cpu_count())
# testloader.num_workers = int(os.cpu_count())

print(f"Number of workers: {trainloader.num_workers}")
print(f"Number of workers: {valloader.num_workers}")
print(f"Number of workers: {testloader.num_workers}")

Number of workers: 8
Number of workers: 8
Number of workers: 8


# Model Training

In [ ]:
# Train the WeatherNet model and record losses
# Use the train function to train the model with optimizer on the trainloader for a specified number of epochs (e.g., 5)
# Record the training and test losses in wn_train_losses and wn_test_losses respectively, pass hyperparameters

# logs to runs/WeatherNet/ or runs/WeatherNetPlusPlus/ or runs/MtlWeatherNet/
writer = SummaryWriter(log_dir=f"runs/{model.name}")

# epochs=1
trainer.train(model, optimizer, trainloader, valloader, epochs, device, "./checkpoints", writer=writer)

writer.flush()

Begin training
Epoch 1/10
* Batch 10/365 - fog loss: 0.06 | glare loss: 0.45 | road loss: 0.51 | traffic loss: 0.31 | weather loss: 1.23 | scene loss: 0.92 | tod loss: 0.37
* Batch 20/365 - fog loss: 0.13 | glare loss: 0.45 | road loss: 0.64 | traffic loss: 0.20 | weather loss: 1.30 | scene loss: 0.84 | tod loss: 0.29
* Batch 30/365 - fog loss: 0.10 | glare loss: 0.48 | road loss: 0.53 | traffic loss: 0.34 | weather loss: 1.22 | scene loss: 0.90 | tod loss: 0.32
* Batch 40/365 - fog loss: 0.01 | glare loss: 0.43 | road loss: 0.42 | traffic loss: 0.26 | weather loss: 1.31 | scene loss: 0.96 | tod loss: 0.31
* Batch 50/365 - fog loss: 0.15 | glare loss: 0.45 | road loss: 0.57 | traffic loss: 0.26 | weather loss: 1.32 | scene loss: 0.91 | tod loss: 0.38
* Batch 60/365 - fog loss: 0.03 | glare loss: 0.46 | road loss: 0.53 | traffic loss: 0.23 | weather loss: 1.24 | scene loss: 0.82 | tod loss: 0.23
* Batch 70/365 - fog loss: 0.08 | glare loss: 0.49 | road loss: 0.56 | traffic loss: 0.32 | 

# (Optional) Test Evaluation

In [ ]:
# Set writer to None to disable TensorBoard logging
trainer.evaluate_model(model, testloader, device, writer=writer)

# Visualize Results

First, ensure you're in the correct environment:

`conda env create -f environment.yml`

This will create a conda environment called *tensorboard*, which you can activate via `conda activate tensorboard`

Then, to visualize the results: 

`tensorboard --logdir=runs`

This will automatically and recursively scan through all run logs in the runs/ directory.